In [1]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import pickle

# Loading the Enhanced and Normalized Dataset

In [2]:
X = pickle.load(open('normalized_enhanced_X', 'rb'))

In [3]:
y = pickle.load(open('y.pkl', 'rb'))

In [4]:
X.shape, y.shape

((426, 256, 256, 3), (426,))

# Train Test Split

In [5]:
from sklearn.model_selection import train_test_split

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [10]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((340, 256, 256, 3), (86, 256, 256, 3), (340,), (86,))

In [8]:
import tensorflow as tf
from tensorflow.keras import layers as L
from keras.layers import *
import keras

## Data Augmentation Layer

In [20]:
data_augmentation = tf.keras.Sequential([
    #RandomCrop(224,224),
    RandomFlip("horizontal"),
    RandomRotation(0.2),
    RandomZoom(0.2, 0.2)
])

# CNN

In [21]:
class SpatialAttentionModule(tf.keras.layers.Layer):
    def __init__(self, kernel_size=3):
        '''
        paper: https://arxiv.org/abs/1807.06521
        code: https://gist.github.com/innat/99888fa8065ecbf3ae2b297e5c10db70
        '''
        super(SpatialAttentionModule, self).__init__()
        self.conv1 = tf.keras.layers.Conv2D(64, kernel_size=kernel_size, 
                                            use_bias=False, 
                                            kernel_initializer='he_normal',
                                            strides=1, padding='same', 
                                            activation=tf.nn.relu)
        self.conv2 = tf.keras.layers.Conv2D(32, kernel_size=kernel_size, 
                                            use_bias=False, 
                                            kernel_initializer='he_normal',
                                            strides=1, padding='same', 
                                            activation=tf.nn.relu)
        self.conv3 = tf.keras.layers.Conv2D(16, kernel_size=kernel_size, 
                                            use_bias=False, 
                                            kernel_initializer='he_normal',
                                            strides=1, padding='same', 
                                            activation=tf.nn.relu)
        self.conv4 = tf.keras.layers.Conv2D(1, kernel_size=kernel_size,  
                                            use_bias=False,
                                            kernel_initializer='he_normal',
                                            strides=1, padding='same', 
                                            activation=tf.math.sigmoid)

    def call(self, inputs):
        avg_out = tf.reduce_mean(inputs, axis=3)
        max_out = tf.reduce_max(inputs,  axis=3)
        x = tf.stack([avg_out, max_out], axis=3) 
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        return self.conv4(x)
    

class ChannelAttentionModule(tf.keras.layers.Layer):
    def __init__(self, ratio=8):
        '''
        paper: https://arxiv.org/abs/1807.06521
        code: https://gist.github.com/innat/99888fa8065ecbf3ae2b297e5c10db70
        '''
        super(ChannelAttentionModule, self).__init__()
        self.ratio = ratio
        self.gapavg = tf.keras.layers.GlobalAveragePooling2D()
        self.gmpmax = tf.keras.layers.GlobalMaxPooling2D()
        
    def build(self, input_shape):
        self.conv1 = tf.keras.layers.Conv2D(input_shape[-1]//self.ratio, 
                                            kernel_size=1, 
                                            strides=1, padding='same',
                                            use_bias=True, activation=tf.nn.relu)
    
        self.conv2 = tf.keras.layers.Conv2D(input_shape[-1], 
                                            kernel_size=1, 
                                            strides=1, padding='same',
                                            use_bias=True, activation=tf.nn.relu)
        super(ChannelAttentionModule, self).build(input_shape)

    def call(self, inputs):
        # compute gap and gmp pooling 
        gapavg = self.gapavg(inputs)
        gmpmax = self.gmpmax(inputs)
        gapavg = tf.keras.layers.Reshape((1, 1, gapavg.shape[1]))(gapavg)   
        gmpmax = tf.keras.layers.Reshape((1, 1, gmpmax.shape[1]))(gmpmax)   
        # forward passing to the respected layers
        gapavg_out = self.conv2(self.conv1(gapavg))
        gmpmax_out = self.conv2(self.conv1(gmpmax))
        return tf.math.sigmoid(gapavg_out + gmpmax_out)
    
    def get_output_shape_for(self, input_shape):
        return self.compute_output_shape(input_shape)

    def compute_output_shape(self, input_shape):
        output_len = input_shape[3]
        return (input_shape[0], output_len)

In [27]:
inputs = Input((256,256,3))
# Convolutional layers
x = Conv2D(32, (3, 3), activation='relu')(inputs)
x = MaxPooling2D((2, 2))(x)
x = Conv2D(64, (3,3), activation='relu')(x)
x = MaxPooling2D((2,2))(x)
x_ca = ChannelAttentionModule()(x)
x_ca = Multiply()([x, x_ca])
x_sa = SpatialAttentionModule()(x)
x_sa = Multiply()([x, x_sa])
x = Concatenate()([x_ca, x_sa])
x = Conv2D(128, (3,3), activation='relu')(x)
x = MaxPooling2D((2,2))(x)
x_ca = ChannelAttentionModule()(x)
x_ca = Multiply()([x, x_ca])
x_sa = SpatialAttentionModule()(x)
x_sa = Multiply()([x, x_sa])
x = Concatenate()([x_ca, x_sa])
x = Conv2D(128, (3,3), activation='relu')(x)
x = MaxPooling2D((2,2))(x)
x = Conv2D(64, (3,3), activation='relu')(x)
x = MaxPooling2D((2,2))(x)
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = BatchNormalization()(x)
x = Dense(64, activation='relu')(x)
x = BatchNormalization()(x)
x = Dense(4, activation='softmax')(x)


cnn = keras.Model(inputs, x)

In [28]:
# Compile the model
cnn.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

In [29]:
class_weights = {0: 1.33125,
                 1: 0.783088,
                 2: 1.745902,
                 3: 1.690476}

In [30]:
cnn.summary()

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_4 (InputLayer)           [(None, 256, 256, 3  0           []                               
                                )]                                                                
                                                                                                  
 conv2d_13 (Conv2D)             (None, 254, 254, 32  896         ['input_4[0][0]']                
                                )                                                                 
                                                                                                  
 max_pooling2d_5 (MaxPooling2D)  (None, 127, 127, 32  0          ['conv2d_13[0][0]']              
                                )                                                           

In [ ]:
history = cnn.fit(x = X_train, 
                  y = y_train,
                  validation_data=[X_test, y_test],
                  class_weight=class_weights,
                  #batch_size=64,
                  epochs=100
                   )

Epoch 1/100
11/11 [==============================] - 55s 4s/step - loss: 1.7385 - accuracy: 0.2324 - val_loss: 1.3963 - val_accuracy: 0.1744
Epoch 2/100
11/11 [==============================] - 43s 4s/step - loss: 1.6840 - accuracy: 0.2706 - val_loss: 1.3965 - val_accuracy: 0.2326
Epoch 3/100
11/11 [==============================] - 43s 4s/step - loss: 1.6372 - accuracy: 0.2941 - val_loss: 1.3978 - val_accuracy: 0.1744
Epoch 4/100
11/11 [==============================] - 43s 4s/step - loss: 1.5777 - accuracy: 0.3382 - val_loss: 1.4008 - val_accuracy: 0.1744
Epoch 5/100
11/11 [==============================] - 43s 4s/step - loss: 1.5392 - accuracy: 0.3882 - val_loss: 1.3880 - val_accuracy: 0.1744
Epoch 6/100
11/11 [==============================] - 42s 4s/step - loss: 1.5128 - accuracy: 0.4088 - val_loss: 1.3809 - val_accuracy: 0.2326
Epoch 7/100
11/11 [==============================] - 42s 4s/step - loss: 1.5231 - accuracy: 0.4147 - val_loss: 1.3823 - val_accuracy: 0.2326
Epoch 8/100
1

# Performance Evaluation

In [86]:
cnn.evaluate(X_test, y_test)

3/3 [==============================] - 1s 401ms/step - loss: 1.4386 - accuracy: 0.3140


[1.4386117458343506, 0.3139534890651703]

In [61]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [87]:
y_pred = cnn.predict(X_test)

3/3 [==============================] - 2s 392ms/step


In [88]:
y_pred = np.argmax(y_pred, axis=1)

In [89]:
y_pred

array([0, 0, 0, 2, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 2,
       0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 2, 0, 2, 2, 0,
       1, 0, 0, 3, 2, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 3, 1, 0, 0,
       3, 0, 0, 1, 2, 1, 0, 0, 0, 0, 2, 0, 0, 0, 0, 1, 0, 0, 1, 0],
      dtype=int64)

In [90]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.23      0.75      0.36        20
           1       0.67      0.17      0.27        35
           2       0.22      0.13      0.17        15
           3       1.00      0.25      0.40        16

    accuracy                           0.31        86
   macro avg       0.53      0.33      0.30        86
weighted avg       0.55      0.31      0.30        86



In [57]:
print(confusion_matrix(y_test, y_pred))

[[13  7  0  0]
 [ 8 23  0  4]
 [10  5  0  0]
 [ 1 13  0  2]]
